In [0]:
# Install required libraries
# azure-storage-blob: connects to Azure Blob Storage
# openpyxl: required by pandas to read .xlsx Excel files
%pip install azure-storage-blob openpyxl -q
print("Libraries installed successfully!")

In [0]:
# Restart kernel after install to load new libraries
dbutils.library.restartPython()

## Read both silver files

In [0]:
# ============================================
# Connect to Azure Blob Storage
# Read both SynergySuite and FocusPOS silver CSVs
# ============================================
from azure.storage.blob import BlobServiceClient
import pandas as pd
import io
import re

# Connect securely via Key Vault
storage_account_key = dbutils.secrets.get(scope="churchs-payroll-kv", key="storage-account-key")
blob_service_client = BlobServiceClient(
    account_url="https://churchspayrollstorage.blob.core.windows.net",
    credential=storage_account_key
)
print("Connected to Azure Blob Storage successfully!")

# Read SynergySuite silver CSV
synergy_blob = blob_service_client.get_blob_client(
    container="silver",
    blob="synergysuite/SynergySuite_Silver.csv"
)
df_synergy = pd.read_csv(io.BytesIO(synergy_blob.download_blob().readall()))
print("SynergySuite silver loaded! Records:", len(df_synergy))

# Read FocusPOS silver CSV
focus_blob = blob_service_client.get_blob_client(
    container="silver",
    blob="focuspos/FocusPOS_Silver.csv"
)
df_focus = pd.read_csv(io.BytesIO(focus_blob.download_blob().readall()))
print("FocusPOS silver loaded! Records:", len(df_focus))

# Validation checks
print("\n=== Validation Checks ===")
assert len(df_synergy) > 0, "ERROR: SynergySuite silver file is empty!"
assert len(df_focus) > 0, "ERROR: FocusPOS silver file is empty!"
print("SynergySuite columns:", df_synergy.columns.tolist())
print("FocusPOS columns:", df_focus.columns.tolist())
print("All validation checks passed!")

## Map Both Sources to Target Schema (12 Columns)

In [0]:
# ============================================
# Map both sources to target schema
# Aggregate by employee per week
# Target: one row per employee per week
# ============================================
import re
import numpy as np
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(message)s")
logger = logging.getLogger(__name__)

def extract_store_id(store_str):
    # Extract numeric store ID from full store name
    # e.g. "FC3607 - S MAIN ST" → "3607"
    # e.g. "Churchs 3978" → "3978"
    match = re.search(r'\d+', str(store_str))
    return match.group() if match else store_str

logger.info("Starting mapping and transformation of both silver sources...")

# Map SynergySuite to target schema
logger.info("Mapping SynergySuite silver to target schema...")
df_synergy_mapped = pd.DataFrame({
    "ID":                 df_synergy["Payroll_ID"],
    "Employee_name":      df_synergy["Employee_name"],
    "StoreID":            df_synergy["StoreID"].apply(extract_store_id),
    "Shift_Date":         pd.to_datetime(df_synergy["Shift_Date"], errors="coerce"),
    "E_Regular_hours":    pd.to_numeric(df_synergy["Regular_Hours"], errors="coerce"),
    "E_overtime_hours":   pd.to_numeric(df_synergy["OT_Hours"], errors="coerce"),
    "E_PTO_Pay_Hours":    np.nan,
    "E_Sick_Pay_Hours":   np.nan,
    "E_Back_Pay_Hours":   np.nan,
    "E_Back_Pay_Dollars": np.nan,
    "E_Tips_Dollars":     pd.to_numeric(df_synergy["Non_Cash_Tips"], errors="coerce").fillna(0) + pd.to_numeric(df_synergy["Declared_Tips"], errors="coerce").fillna(0),
    "week_end_date":      pd.to_datetime(df_synergy["week_end_date"], errors="coerce")
})
logger.info(f"SynergySuite mapped records: {len(df_synergy_mapped)}")
print("SynergySuite mapped records:", len(df_synergy_mapped))

# Map FocusPOS to target schema
# FocusPOS Hours is total hours — split into Regular and OT
# Regular = hours up to 8 per shift, OT = hours over 8
logger.info("Mapping FocusPOS silver to target schema...")
logger.info("FocusPOS OT calculation: Regular = min(Hours, 8), OT = max(0, Hours - 8)")
df_focus_mapped = pd.DataFrame({
    "ID":                 df_focus["Payroll_ID"],
    "Employee_name":      df_focus["Employee_name"],
    "StoreID":            df_focus["StoreID"].apply(extract_store_id),
    "Shift_Date":         pd.to_datetime(df_focus["Shift_Date"], errors="coerce"),
    "E_Regular_hours":    pd.to_numeric(df_focus["Hours"], errors="coerce").apply(
                              lambda h: min(h, 8) if pd.notna(h) else np.nan),
    "E_overtime_hours":   pd.to_numeric(df_focus["Hours"], errors="coerce").apply(
                              lambda h: max(0, h - 8) if pd.notna(h) else np.nan),
    "E_PTO_Pay_Hours":    np.nan,
    "E_Sick_Pay_Hours":   np.nan,
    "E_Back_Pay_Hours":   np.nan,
    "E_Back_Pay_Dollars": np.nan,
    "E_Tips_Dollars":     pd.to_numeric(df_focus["Declared_Tips"], errors="coerce").fillna(0),
    "week_end_date":      pd.to_datetime(df_focus["week_end_date"], errors="coerce")
})
logger.info(f"FocusPOS mapped records: {len(df_focus_mapped)}")
print("FocusPOS mapped records:", len(df_focus_mapped))

# Merge both sources into one DataFrame
logger.info("Concatenating both sources into one DataFrame...")
df_combined = pd.concat([df_synergy_mapped, df_focus_mapped], ignore_index=True)
logger.info(f"Combined records before aggregation: {len(df_combined)}")
print("Combined records before aggregation:", len(df_combined))

# Aggregate by Employee + Week — one row per employee per week
logger.info("Aggregating by Employee + Store + Week...")
df_merged = df_combined.groupby(
    ["ID", "Employee_name", "StoreID", "week_end_date"],
    as_index=False
).agg({
    "E_Regular_hours":    "sum",
    "E_overtime_hours":   "sum",
    "E_PTO_Pay_Hours":    "sum",
    "E_Sick_Pay_Hours":   "sum",
    "E_Back_Pay_Hours":   "sum",
    "E_Back_Pay_Dollars": "sum",
    "E_Tips_Dollars":     "sum"
})
logger.info(f"Records after aggregation: {len(df_merged)}")
print("Records after aggregation:", len(df_merged))

# These columns have no source data — leave as empty string (shows as empty in CSV)
logger.info("Setting PTO/Sick/Back Pay columns to empty — no source data available")
df_merged["E_PTO_Pay_Hours"]    = ""
df_merged["E_Sick_Pay_Hours"]   = ""
df_merged["E_Back_Pay_Hours"]   = ""
df_merged["E_Back_Pay_Dollars"] = ""

# Round numeric columns to 2 decimal places
logger.info("Rounding numeric columns to 2 decimal places...")
df_merged["E_Regular_hours"]  = df_merged["E_Regular_hours"].round(2)
df_merged["E_overtime_hours"] = df_merged["E_overtime_hours"].round(2)
df_merged["E_Tips_Dollars"]   = df_merged["E_Tips_Dollars"].round(2)

# Check invalid IDs before cleaning
logger.info("Checking for invalid IDs (system placeholders)...")
invalid = df_merged[pd.to_numeric(df_merged["ID"], errors="coerce").isna()]
print("\nInvalid ID rows (system placeholders):")
print(invalid[["ID", "Employee_name", "StoreID"]].to_string())
logger.info(f"Total invalid rows to remove: {len(invalid)}")
print("Total invalid rows to remove:", len(invalid))

# Remove system placeholder rows where ID is dash
# 'Taker, Order' is a POS system placeholder not a real employee
df_merged["ID"] = pd.to_numeric(df_merged["ID"], errors="coerce")
df_merged = df_merged.dropna(subset=["ID"])
df_merged["ID"] = df_merged["ID"].astype(int)
logger.info(f"Records after removing invalid IDs: {len(df_merged)}")
print("Records after removing invalid IDs:", len(df_merged))

# Calculate relative week number based on earliest week_end_date
# Week 1 = first week of pay period, Week 2 = second week etc.
logger.info("Calculating relative week numbers...")
min_week = df_merged["week_end_date"].min()
df_merged["Week"] = df_merged["week_end_date"].apply(
    lambda d: ((pd.to_datetime(d) - pd.to_datetime(min_week)).days // 7) + 1
)
logger.info(f"Week numbers assigned: {sorted(df_merged['Week'].unique().tolist())}")
print("Week numbers assigned:", sorted(df_merged["Week"].unique().tolist()))

# Format week_end_date to M/D/YYYY to match target format
logger.info("Formatting week_end_date to M/D/YYYY...")
df_merged["week_end_date"] = df_merged["week_end_date"].dt.strftime("%-m/%-d/%Y")

# Reorder columns to match target schema
logger.info("Reordering columns to match target schema...")
df_merged = df_merged[[
    "ID", "Employee_name", "StoreID", "Week",
    "E_Regular_hours", "E_overtime_hours",
    "E_PTO_Pay_Hours", "E_Sick_Pay_Hours",
    "E_Back_Pay_Hours", "E_Back_Pay_Dollars",
    "E_Tips_Dollars", "week_end_date"
]]

# Validation checks
print("\n=== Validation Checks ===")
try:
    assert len(df_merged) > 0, "ERROR: Merged DataFrame is empty!"
    assert df_merged["ID"].notna().all(), "ERROR: Null IDs found!"
    assert df_merged["Employee_name"].notna().all(), "ERROR: Null employee names!"
    assert df_merged["StoreID"].notna().all(), "ERROR: Null store IDs!"
    assert df_merged["week_end_date"].notna().all(), "ERROR: Null week end dates!"
    assert df_merged["Week"].isin([1, 2, 3]).all(), "ERROR: Invalid week numbers!"
    assert (df_merged["E_Regular_hours"] >= 0).all(), "ERROR: Negative regular hours!"
    assert (df_merged["E_overtime_hours"] >= 0).all(), "ERROR: Negative OT hours!"
    logger.info("All validation checks passed!")
    print("All validation checks passed!")
except AssertionError as e:
    logger.error(f"VALIDATION FAILED: {e}")
    raise

print("\n=== Final Summary ===")
print("Total records:", len(df_merged))
print("Unique employees:", df_merged["ID"].nunique())
print("Unique stores:", df_merged["StoreID"].nunique())
print("Weeks covered:", sorted(df_merged["Week"].unique().tolist()))
print("Regular hours range:", df_merged["E_Regular_hours"].min(), "to", df_merged["E_Regular_hours"].max())
print("OT hours range:", df_merged["E_overtime_hours"].min(), "to", df_merged["E_overtime_hours"].max())
print("Sample week_end_date format:", df_merged["week_end_date"].iloc[0])
logger.info(f"Final merged records: {len(df_merged)} | Employees: {df_merged['ID'].nunique()} | Stores: {df_merged['StoreID'].nunique()}")
print("\nSample:")
print(df_merged.head(10).to_string())

## Write Merged Data To Gold Container

In [0]:
# ============================================
# Write merged payroll data to gold container
# ADF will pick this up and load into SQL Database
# ============================================

logger.info("Starting upload of merged payroll data to gold container...")

# Convert DataFrame to CSV in memory
csv_buffer = io.BytesIO()
df_merged.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)
logger.info(f"CSV buffer created — {len(df_merged)} records ready for upload")

# Upload to gold container
gold_client = blob_service_client.get_blob_client(
    container="gold",
    blob="payroll/merged_payroll.csv"
)

try:
    gold_client.upload_blob(csv_buffer, overwrite=True)
    logger.info("Merged payroll CSV uploaded to gold container successfully!")
    logger.info("Location: gold/payroll/merged_payroll.csv")
except Exception as e:
    logger.error(f"ERROR: Failed to upload merged payroll CSV to gold — {e}")
    raise

# Validation — verify upload
try:
    assert len(df_merged) > 0, "ERROR: Nothing was written to gold!"
    logger.info("Gold upload validation passed!")
    print("\nMerged data written to gold container!")
    print("Location: gold/payroll/merged_payroll.csv")
    print("Total records written:", len(df_merged))
    print("Columns written:", df_merged.columns.tolist())
    print("Upload validation passed!")
except AssertionError as e:
    logger.error(f"GOLD UPLOAD VALIDATION FAILED: {e}")
    raise

logger.info("Notebook 3 completed successfully — silver merge and gold upload done!")

## Validate SQL Database Connection

In [0]:
# ============================================
# Validate SQL Database connection from Databricks
# This confirms the database is reachable before ADF loads data
# ============================================

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(message)s")
logger = logging.getLogger(__name__)

logger.info("Installing pymssql library...")
%pip install pymssql

import pymssql

logger.info("Attempting SQL Database connection...")
logger.info("Server: churchs-payroll-server.database.windows.net")
logger.info("Database: churchs-payroll-db")
logger.info("User: payrolladmin")

try:
    conn = pymssql.connect(
        server="churchs-payroll-server.database.windows.net",
        user="payrolladmin",
        password=dbutils.secrets.get(scope="churchs-payroll-kv", key="payrolladmiin-secret"),
        database="churchs-payroll-db"
    )
    logger.info("SQL Database connection successful!")
    print("Connected successfully!")
    conn.close()
    logger.info("SQL Database connection closed successfully!")
except pymssql.OperationalError as e:
    logger.error(f"ERROR: Failed to connect to SQL Database — {e}")
    raise
except Exception as e:
    logger.error(f"ERROR: Unexpected error during SQL connection — {e}")
    raise